# SDS benchmark — heavy runs on the IDH hub

End-to-end run of the S4 baselines battery + the pointwise judge on
`Qwen/Qwen3.5-2B` served by vLLM. **Run All.** Everything is written under
`persistent_volume/` (repo, HF cache, results), so it survives the session.

Set the switches in the last cell: `JOB` = `smoke` | `battery` | `judge` | `all`,
and `BASE_URL` if a vLLM server is already running.


In [ ]:
import subprocess, sys
print("python:", sys.version.split()[0])
for cmd in (["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"],
            ["free", "-h"], ["df", "-h", "."]):
    try:
        print("$", " ".join(cmd))
        print(subprocess.run(cmd, capture_output=True, text=True).stdout.strip(), "\n")
    except FileNotFoundError:
        print("$", cmd[0], ": not found\n")


In [ ]:
import os, subprocess, pathlib
PERSIST = next((p for p in [pathlib.Path.cwd()/"persistent_volume",
                            pathlib.Path.home()/"persistent_volume",
                            pathlib.Path("/content/persistent_volume")] if p.is_dir()), None)
WORK = ((PERSIST or pathlib.Path.home())/"sds-hub"); WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK/"Evidence-Grounded-Document-Extraction-for-Regulated-Workflows"
URL = "https://github.com/FaritSharafutdinov/Evidence-Grounded-Document-Extraction-for-Regulated-Workflows.git"
BRANCH = os.environ.get("BRANCH", "farit_dev")
if (REPO/".git").exists():
    for a in (["fetch","origin",BRANCH], ["checkout",BRANCH], ["pull","--ff-only","origin",BRANCH]):
        subprocess.run(["git","-C",str(REPO),*a])
else:
    subprocess.run(["git","clone","-b",BRANCH,URL,str(REPO)], check=True)
head = subprocess.run(["git","-C",str(REPO),"rev-parse","--short","HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("WORKDIR:", WORK)
print("repo:", REPO, "@", head)


In [ ]:
import os, subprocess
# ---- switches -------------------------------------------------------------
os.environ.setdefault("JOB", "all")            # smoke | battery | judge | all
os.environ.setdefault("MODEL", "Qwen/Qwen3.5-2B")
os.environ.setdefault("LAYER", "both")
os.environ.setdefault("LAUNCH_VLLM", "auto")   # auto | always | never
os.environ.setdefault("INSTALL_VLLM", "1")     # pip install vllm if missing
# os.environ["BASE_URL"] = "http://127.0.0.1:8000"  # if a server is already up
# ---------------------------------------------------------------------------
env = os.environ.copy()
env["WORKDIR"] = str(WORK)
env["HF_HOME"] = str(WORK/"hf")
subprocess.run(["bash", str(REPO/"farit/hub/run_on_hub.sh")], cwd=str(REPO), env=env)


In [ ]:
import shutil, pathlib
out = WORK/"out"
if out.is_dir() and any(out.iterdir()):
    z = shutil.make_archive(str(WORK/"hub-results"), "zip", root_dir=str(out))
    print("Open the file browser (left panel) and download:", z)
else:
    print("no out/ yet — run the driver cell first")
